# 무_이상치제거_주간기준_등급코드 EDA

머신러닝/딥러닝 분석을 위한 EDA(탐색적 데이터 분석)를 진행합니다.
- 데이터 파일: 무_이상치제거_주간기준_등급코드 - 복사본.csv
- 주요 단계: 라이브러리 임포트, 데이터 불러오기, 구조 및 통계 확인, 샘플 미리보기

In [ ]:
# 필요 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# CSV 파일 불러오기
csv_path = r'c:\ai_x\source\JikFam\data\무_월차_작년평균단가포함.csv'
df = pd.read_csv(csv_path, encoding='cp949')
df.shape

In [ ]:
# 데이터프레임 기본 정보 확인
print('행/열 개수:', df.shape)
print('컬럼명:', df.columns.tolist())
df.info()

In [ ]:
# 컬럼별 데이터 타입 및 결측치 확인
print('컬럼별 데이터 타입:')
print(df.dtypes)
print('\n컬럼별 결측치 개수:')
print(df.isnull().sum())

In [ ]:
# 컬럼별 주요 통계량 확인
df.describe(include='all')

In [ ]:
# 샘플 데이터 미리보기
print('--- head() ---')
print(df.head())
print('\n--- tail() ---')
print(df.tail())

In [ ]:
# 분석 제외 컬럼 및 결측치 제거
exclude_cols = ['holiday_flag', 'holiday_score', 'grow_score']

df_clean = df.drop(columns=exclude_cols)
# 결측치가 있는 행 제거
print('제거 전 shape:', df_clean.shape)
df_clean = df_clean.dropna()
print('제거 후 shape:', df_clean.shape)
df_clean.info()

## AI 모델링을 위한 EDA 분석 포인트 요약

- 주차 → 시작일 기준 datetime 변환 (시계열 분석)
- 품종명, 등급이름, 등급코드로 그룹화하여 비교 분석
- 기온, 강수량, 습도와 가격(평균단가, 주간평균단가) 간의 상관관계 분석

이외에도 AI 모델링에 필요한 데이터 분포, 이상치, 타겟/피처 정의 등도 함께 확인합니다.

In [ ]:
# 주차 컬럼에서 시작일(앞부분)만 추출하여 datetime으로 변환
df_clean['week_start'] = pd.to_datetime(df_clean['week'].str.split('~').str[0], format='%Y-%m-%d', errors='coerce')
print(df_clean[['week', 'week_start']].head())

In [ ]:
# 품종명, 등급이름, 등급코드별 가격 통계(평균) 비교
group_cols = ['품종명', '등급이름', '등급코드']
price_cols = ['평균단가(원)', '주간평균단가(원)']
grouped = df_clean.groupby(group_cols)[price_cols].mean().reset_index()
print(grouped.head())

In [ ]:
# matplotlib 한글 폰트 설정 (Windows 환경)
import matplotlib.font_manager as fm
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 기온, 강수량, 습도와 가격 간의 상관관계 분석
weather_cols = ['일평균기온', '최고기온', '최저기온', '평균상대습도', '강수량(mm)', '1시간최고강수량(mm)']
target_cols = ['평균단가(원)', '주간평균단가(원)']
cor_df = df_clean[weather_cols + target_cols].corr()
print(cor_df[target_cols])
# 시각화
plt.figure(figsize=(8,5))
sns.heatmap(cor_df[target_cols], annot=True, cmap='coolwarm')
plt.title('기상 변수와 가격 간 상관관계')
plt.show()

## 실무에서 많이 사용하는 추가 EDA 예시

- 타겟(가격) 분포 시각화
- 수치형/범주형 변수별 분포 및 이상치 탐색
- 상관관계 히트맵
- 피처별 타겟 평균 비교
- (필요시) 변수간 pairplot 등

In [ ]:
# 주차(시계열)별 평균단가 변동 시각화
# 주차_시작일 기준으로 평균단가(원) 시계열 그래프
plt.figure(figsize=(14,5))
df_clean_sorted = df_clean.sort_values('주차_시작일')
plt.plot(df_clean_sorted['주차_시작일'], df_clean_sorted['평균단가(원)'], marker='o', linestyle='-', alpha=0.5, label='평균단가(원)')
plt.title('주차별 평균단가(원) 시계열 변동')
plt.xlabel('주차 시작일')
plt.ylabel('평균단가(원)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 주차별(주차_시작일 기준) 평균 가격 변동 시각화
weekly_price = df_clean.groupby('주차_시작일')['평균단가(원)'].mean().reset_index()
plt.figure(figsize=(14,5))
plt.plot(weekly_price['주차_시작일'], weekly_price['평균단가(원)'], marker='o', linestyle='-', alpha=0.7, label='주차별 평균단가(원)')
plt.title('주차별 평균단가(원) 시계열 변동 (집계)')
plt.xlabel('주차 시작일')
plt.ylabel('평균단가(원)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 품종별 주차(시작일)별 평균단가(원) 시계열 그래프 (한 화면, 선 여러개)
plt.figure(figsize=(14,6))
품종_list = df_clean['품종명'].unique()
for 품종 in 품종_list:
    temp = df_clean[df_clean['품종명'] == 품종]
    temp_group = temp.groupby('주차_시작일')['평균단가(원)'].mean().reset_index()
    plt.plot(temp_group['주차_시작일'], temp_group['평균단가(원)'], marker='o', linestyle='-', label=품종)
plt.title('품종별 주차별 평균단가(원) 시계열 추이')
plt.xlabel('주차 시작일')
plt.ylabel('평균단가(원)')
plt.legend(title='품종명')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# RandomForestRegressor를 활용한 피처 중요도 평가
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

# 타겟과 피처 분리 (평균단가 예측)
target = '평균단가(원)'
features = df_clean.drop(columns=[target, '주간평균단가(원)', '주차', '주차_시작일'])

# 범주형 변수 인코딩
for col in features.select_dtypes(include='object').columns:
    features[col] = LabelEncoder().fit_transform(features[col])

X = features
y = df_clean[target]

# 랜덤포레스트 모델 학습
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

# 중요도 시각화
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10,5))
plt.title('Feature Importances (RandomForest)')
plt.bar(range(X.shape[1]), importances[indices], align='center')
plt.xticks(range(X.shape[1]), X.columns[indices], rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 불필요 컬럼 제외 후 중요도 평가 (RandomForestRegressor)
drop_cols = [
    '평균단가(원)', '주간평균단가(원)', '주차', '주차_시작일', '연월일', '총금액(원)', '총거래량(kg)'
]
features = df_clean.drop(columns=drop_cols)

# 범주형 변수 인코딩
for col in features.select_dtypes(include='object').columns:
    features[col] = LabelEncoder().fit_transform(features[col])

X = features
y = df_clean['평균단가(원)']

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10,5))
plt.title('Feature Importances (RandomForest, 불필요 컬럼 제외)')
plt.bar(range(X.shape[1]), importances[indices], align='center')
plt.xticks(range(X.shape[1]), X.columns[indices], rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 품종명별 데이터 비율(%) 계산 및 출력
var_counts = df_clean['품종명'].value_counts(normalize=True) * 100
for 품종, 비율 in var_counts.items():
    print(f'{품종}: {비율:.1f}%')

In [ ]:
# 품종명별 비율 파이차트 시각화
var_counts = df_clean['품종명'].value_counts(normalize=True) * 100
plt.figure(figsize=(7,7))
plt.pie(var_counts, labels=var_counts.index, autopct='%1.1f%%', startangle=140, counterclock=False)
plt.title('품종명별 데이터 비율')
plt.axis('equal')
plt.show()

In [ ]:
# 의미 있는 컬럼만 골라 상관관계 히트맵 시각화 (실무형)
# 제외할 컬럼 정의
exclude_cols = [
    '평균단가(원)', '주간평균단가(원)', '주차', '주차_시작일', '연월일', '총금액(원)', '총거래량(kg)',
    '품목코드', '품목명', '등급이름',
]
# 수치형 변수만 추출 (의미 있는 컬럼만)
num_cols = [col for col in df_clean.select_dtypes(include=[np.number]).columns if col not in exclude_cols]
# 타겟 포함
corr_cols = num_cols + ['평균단가(원)']
corr = df_clean[corr_cols].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5, cbar_kws={"shrink":.8})
plt.title('상관관계 히트맵')
plt.tight_layout()
plt.show()

In [ ]:
# 수치형 변수 히스토그램
num_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in num_cols:
    plt.figure(figsize=(6,2))
    sns.histplot(df_clean[col], bins=50, kde=True)
    plt.title(f'{col} 분포')
    plt.show()